In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "finalyear-project-rev2"
REPO_URL = "https://github.com/AmarjithTK/finalyear-project-rev2.git"
repo_path = Path("/content") / REPO

# Always escape to safe directory first
os.chdir("/content")

if not repo_path.exists():
    print("Cloning repository...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(repo_path)],
        check=True
    )
else:
    print("Repo exists. Resetting and pulling latest...")

    os.chdir(repo_path)

    # Force discard all local changes
    subprocess.run(["git", "reset", "--hard", "HEAD"], check=True)

    # Clean untracked files (VERY important)
    subprocess.run(["git", "clean", "-fd"], check=True)

    # Pull latest changes
    subprocess.run(["git", "pull"], check=True)

# Enter repo (important if just cloned)
os.chdir(repo_path)

basedir = str(repo_path) + os.sep

print("Current directory:", os.getcwd())
print("Base directory set to:", basedir)

data_path = os.path.join(basedir, "data", "microgrid.csv")
print("Data path:", data_path)

: 

In [ ]:
import pandas as pd
import os

# Construct the file path using the basedir defined in the previous cell
data_path = os.path.join(basedir, 'data/microgrid.csv')

try:
    # Read the CSV file and display the first few rows
    df = pd.read_csv(data_path)
    display(df.head())
except Exception as e:
    print(f"Failed to read CSV at {data_path}: {e}")

In [ ]:
# 1. Execute Training & Prediction Scripts
# Handled via the .py scripts in src/
# If in Colab, this will execute the cloned python modules.
# Optional dependency install in Colab:
# !pip install -r {basedir}requirements.txt

!cd {basedir} && python -m src.train
!cd {basedir} && python -m src.predict
!cd {basedir} && python -m src.scenarios

In [ ]:
# 2. Load Outputs
import json

try:
    preds = pd.read_csv(os.path.join(basedir, "outputs/predictions.csv"))
    preds["timestamp"] = pd.to_datetime(preds["timestamp"])
    
    with open(os.path.join(basedir, "outputs/metrics.json")) as f:
        metrics = json.load(f)
        
    with open(os.path.join(basedir, "outputs/scenarios.json")) as f:
        scenarios = json.load(f)
        
    print("Files loaded successfully!")
    print("\nMetrics:")
    print(metrics)
except Exception as e:
    print(f"Failed to load outputs: {e}")

In [ ]:
# 3. Visualize Predictions
import matplotlib.pyplot as plt

if 'preds' in locals():
    pred_cols = [c for c in preds.columns if c.startswith("predicted_")]
    if not pred_cols:
        print("No predicted_* columns found in outputs/predictions.csv")
    else:
        for pred_col in pred_cols:
            target = pred_col.replace("predicted_", "")
            actual_col = f"actual_{target}"

            plt.figure(figsize=(10, 5))
            if actual_col in preds.columns:
                plt.plot(preds["timestamp"], preds[actual_col], label=f"Actual {target}", marker='o')
            plt.plot(preds["timestamp"], preds[pred_col], label=f"Predicted {target}", marker='x')
            plt.title(f"Actual vs Predicted {target}")
            plt.xlabel("Time")
            plt.ylabel(target)
            plt.legend()
            plt.grid(True)
            plt.show()

In [ ]:
# 4. Scenario Assessment
if 'scenarios' in locals():
    print("=== Scenario Assessment ===")
    for scenario_name, details in scenarios.items():
        print(f"\n{scenario_name.replace('_', ' ')}:")
        for key, value in details.items():
            print(f"  - {key.replace('_', ' ').capitalize()}: {value}")